# HGB·CatBoost 66개 LOFO + Trackman 제구 피처 실험

Google Drive의 원본 CSV를 직접 읽습니다. 데이터는 GitHub에 업로드하지 않으며, 각 모델 실행 직후 결과를 Drive에 체크포인트로 저장합니다.

- 기준선: 공식 기본 47 + 팀원 중요도 6 + 과거 추세 13 = 66개
- 검증: 2024 홀드아웃(학습은 2023년까지)
- 중요도: 피처 제거 후 Brier 악화량(LOFO)
- Trackman: 세이버메트릭 그룹/개별 피처 추가 후 Brier 개선량

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 설정

먼저 `quick`으로 경로와 실행을 확인하세요. 최종 결론은 `full`로 다시 실행해야 합니다. `TRACKMAN_SCOPE='groups'`는 6개 그룹만, `'all'`은 그룹과 모든 개별 Trackman 피처를 실행합니다.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tswaincae1221/lg_aimers_experiment_lab.git'
BRANCH = 'agent/compact-features-hgb-catboost'
PROJECT_DIR = Path('/content/lg_aimers_experiment_lab')
DATA_DIR = Path('/content/drive/MyDrive/aimers_data')

MODE = 'quick'                 # quick: 점검용, full: 최종 결과
PHASE = 'all'                  # all / lofo / trackman
TRACKMAN_SCOPE = 'all'         # none / groups / all
CATBOOST_TASK_TYPE = 'CPU'     # GPU 런타임이면 GPU로 변경 가능
N_JOBS = 4

TRAIN_PATH = DATA_DIR / 'train.csv'
TRACKMAN_PATH = DATA_DIR / 'trackman_history.csv'
OUTPUT_DIR = DATA_DIR / 'results' / 'lofo_trackman' / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

In [ ]:
import subprocess

if not (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt'),
     '-r', str(PROJECT_DIR / 'requirements-optional.txt')],
    check=True,
)

## Drive 파일 확인

아래 셀은 다운로드하지 않고 마운트된 Drive 경로에 파일이 있는지만 확인합니다.

In [ ]:
required = [TRAIN_PATH, TRACKMAN_PATH, PROJECT_DIR / 'resources/pitcher_trackman_mapping.csv']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('파일 경로를 확인하세요: ' + ', '.join(missing))
[(path.name, round(path.stat().st_size / 1024**2, 1)) for path in required]

## 실험 실행

같은 설정으로 재실행하면 `experiment_history_{mode}.csv`를 확인해 완료된 모델은 건너뜁니다. Colab 런타임이 끊겨도 이 셀만 다시 실행하면 이어집니다.

In [ ]:
import os
import sys

command = [
    sys.executable, '-m', 'src.lofo_trackman_runner',
    '--train', str(TRAIN_PATH),
    '--trackman', str(TRACKMAN_PATH),
    '--mapping', str(PROJECT_DIR / 'resources/pitcher_trackman_mapping.csv'),
    '--output-dir', str(OUTPUT_DIR),
    '--mode', MODE,
    '--phase', PHASE,
    '--trackman-scope', TRACKMAN_SCOPE,
    '--models', 'hist_gbdt', 'catboost',
    '--catboost-task-type', CATBOOST_TASK_TYPE,
    '--n-jobs', str(N_JOBS),
]
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True, env=os.environ.copy())

## 결과 표

LOFO의 `delta_brier`는 제거 후 악화량, Trackman의 `brier_improvement`는 추가 후 개선량입니다. 둘 다 양수가 클수록 좋습니다.

In [ ]:
import pandas as pd
from IPython.display import display

baseline = pd.read_csv(OUTPUT_DIR / 'baseline_scores.csv')
lofo = pd.read_csv(OUTPUT_DIR / 'lofo_importance_combined.csv')
trackman = pd.read_csv(OUTPUT_DIR / 'trackman_addback_combined.csv')

print('기준선')
baseline_columns = [
    column for column in ['model', 'brier', 'auc', 'ece_10bin', 'elapsed_seconds']
    if column in baseline.columns
]
display(baseline[baseline_columns])
print('LOFO 상위 20개')
display(lofo.head(20))
print('Trackman 그룹')
display(trackman[trackman['experiment_type'] == 'trackman_group'])
print('Trackman 개별 상위 20개')
display(trackman[trackman['experiment_type'] == 'trackman_individual'].head(20))
print('합의 상위 Trackman 묶음(탐색적 확인)')
display(trackman[trackman['experiment_type'] == 'trackman_selected_bundle'])

In [ ]:
import matplotlib.pyplot as plt

top = lofo.head(20).sort_values('mean_delta_brier')
colors = top['feature_group'].map({
    'basic': '#4C78A8', 'teammate_important': '#F58518', 'trend': '#54A24B'
})
plt.figure(figsize=(10, 7))
plt.barh(top['feature'], top['mean_delta_brier'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Mean Brier increase after removal (higher = more important)')
plt.title(f'{MODE}: HGB + CatBoost combined LOFO top 20')
plt.tight_layout()
plt.show()

In [ ]:
tm_group = trackman[trackman['experiment_type'] == 'trackman_group'].copy()
tm_group = tm_group.sort_values('mean_brier_improvement')
plt.figure(figsize=(9, 4.5))
plt.barh(tm_group['item'], tm_group['mean_brier_improvement'], color='#8C6BB1')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Mean Brier improvement after add-back (higher = better)')
plt.title(f'{MODE}: Trackman sabermetric group value')
plt.tight_layout()
plt.show()

## 최종 실행 전 체크

`quick` 결과는 코드와 방향 확인용입니다. 최종 수치를 만들 때 설정 셀에서 `MODE='full'`, `TRACKMAN_SCOPE='all'`로 바꾸고 아래 셀부터 다시 실행하세요. full 결과는 별도 폴더에 저장되므로 quick 체크포인트와 섞이지 않습니다.